# 🔗 Notebook 02 — Link Extraction & Defanging Module
### Phishing Email Forensic Analyzer & Link Defanger (Part 2 of 4)

This notebook:
1. Mounts Drive and installs `tldextract`.
2. Writes **`link_defanger.py`** into `modules/`.
3. Demonstrates URL extraction, **defanging**, and **sender-vs-link
   domain mismatch detection**.

> **Cybersecurity context — defanging:** Analysts routinely share
> malicious URLs in reports, tickets, or chat with colleagues. If a
> real `https://` link is pasted somewhere with link-previews enabled,
> it can auto-fetch and sometimes even trigger the payload. *Defanging*
> (`http` → `hxxp`, `.` → `[.]`) neutralizes the URL so it displays as
> plain text everywhere, while remaining perfectly readable to a human.
>
> **Cybersecurity context — domain mismatch:** The single highest-value
> phishing signal is a mismatch between the sender's claimed
> organization and where its links actually go (e.g. mail claiming to
> be PayPal linking to `paypa1-verify.com`). Comparing raw strings is
> useless because subdomains (`secure.paypal.com.evil.co`) or
> shorteners can fool naive checks — we use `tldextract` to reliably
> pull out the *registered* domain (domain + public suffix) for a fair
> comparison.

## 1. Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tldextract

import os
PROJECT_ROOT = "/content/drive/MyDrive/Phishing_Analyzer_Project"
MODULES_DIR = os.path.join(PROJECT_ROOT, "modules")
os.makedirs(MODULES_DIR, exist_ok=True)
print("Modules dir:", MODULES_DIR)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 5.6 MB/s eta 0:00:00
Modules dir: /content/drive/MyDrive/Phishing_Analyzer_Project/modules


## 2. Write `link_defanger.py` to Drive

In [2]:
%%writefile /content/drive/MyDrive/Phishing_Analyzer_Project/modules/link_defanger.py


import re
import email.utils
import tldextract

# Matches http(s):// URLs and bare "www." links; stops at whitespace or
# common trailing/wrapping characters so we don't swallow punctuation.
URL_REGEX = re.compile(r"""(?i)\b((?:https?://|www\.)[^\s<>"'\)\]]+)""")


def extract_clean_email_address(header_value):
    """
    Headers like `From:` often arrive as a display name plus address,
    e.g.  "PayPal Security" <service@paypal.com>
    Feeding that raw string into domain extraction would risk picking
    up junk from the display name or trailing angle brackets, so we
    use the standard library's RFC 2822 address parser to reliably
    pull out just the bare email address first.
    """
    if not header_value:
        return ""
    _, address = email.utils.parseaddr(header_value)
    return address


def extract_urls(text):
    """Return a de-duplicated, order-preserving list of URLs found in text."""
    if not text:
        return []
    raw_matches = URL_REGEX.findall(text)
    cleaned = []
    for u in raw_matches:
        # Strip common trailing punctuation that regex greedily includes
        u = u.rstrip(".,;:!?)\'\"")
        if u:
            cleaned.append(u)
    # de-dup while preserving first-seen order
    seen = set()
    unique = []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            unique.append(u)
    return unique


def defang_url(url):
    """
    Neutralize a URL for safe display/sharing:
      http://  -> hxxp://
      https:// -> hxxps://
      .        -> [.]
    """
    if not url:
        return url
    defanged = url.replace("https://", "hxxps://").replace("http://", "hxxp://")
    defanged = defanged.replace(".", "[.]")
    return defanged


def defang_email_address(address):
    """Defang an email address the same way (@ -> [at], . -> [.])."""
    if not address:
        return address
    return address.replace("@", "[at]").replace(".", "[.]")


def get_registered_domain(url_or_address):
    """
    Return the registrable domain (domain + public suffix, e.g.
    'paypal.com' or 'bbc.co.uk') for a URL, bare domain, or email-like
    string. This is the correct unit of comparison for brand
    impersonation checks -- comparing full hostnames or raw substrings
    is easily defeated by subdomains.
    """
    if not url_or_address:
        return ""
    extracted = tldextract.extract(url_or_address)
    if extracted.domain and extracted.suffix:
        return f"{extracted.domain}.{extracted.suffix}"
    return extracted.domain or ""


def analyze_links(body_text, sender_address):
    """
    Extract every URL in body_text, defang it, and flag whether its
    registered domain differs from the sender's claimed registered
    domain.

    Returns a list of dicts:
        {
          "original_url": ...,
          "defanged_url": ...,
          "link_domain": ...,
          "sender_domain": ...,
          "domain_mismatch": bool,
        }
    """
    clean_sender = extract_clean_email_address(sender_address)
    sender_domain = get_registered_domain(clean_sender)
    urls = extract_urls(body_text)

    results = []
    for url in urls:
        link_domain = get_registered_domain(url)
        mismatch = bool(sender_domain) and bool(link_domain) and (link_domain != sender_domain)
        results.append({
            "original_url": url,
            "defanged_url": defang_url(url),
            "link_domain": link_domain,
            "sender_domain": sender_domain,
            "domain_mismatch": mismatch,
        })
    return results


Overwriting /content/drive/MyDrive/Phishing_Analyzer_Project/modules/link_defanger.py


## 3. Self-Test — Extraction, Defanging & Mismatch Detection

In [3]:
import sys
sys.path.append(MODULES_DIR)

import link_defanger
import importlib
importlib.reload(link_defanger)

sample_body = '''
Dear Customer, please verify your account immediately at
http://paypa1-verify.com/login before it is suspended.
For reference, our real help center is at https://www.paypal.com/help.
'''

sender = '"PayPal Security" <service@paypal.com>'  # realistic raw From: header

print("--- Extracted URLs ---")
urls = link_defanger.extract_urls(sample_body)
for u in urls:
    print(" ", u)

print("\n--- Defanged versions ---")
for u in urls:
    print(" ", link_defanger.defang_url(u))

print("\n--- Domain mismatch analysis (sender: {}) ---".format(sender))
results = link_defanger.analyze_links(sample_body, sender)
for r in results:
    flag = " MISMATCH" if r["domain_mismatch"] else "OK"
    print(f" [{flag}] {r['link_domain']:<20} (sender: {r['sender_domain']})  -> {r['defanged_url']}")

--- Extracted URLs ---
  http://paypa1-verify.com/login
  https://www.paypal.com/help

--- Defanged versions ---
  hxxp://paypa1-verify[.]com/login
  hxxps://www[.]paypal[.]com/help

--- Domain mismatch analysis (sender: "PayPal Security" <service@paypal.com>) ---
 [ MISMATCH] paypa1-verify.com    (sender: paypal.com)  -> hxxp://paypa1-verify[.]com/login
 [OK] paypal.com           (sender: paypal.com)  -> hxxps://www[.]paypal[.]com/help
